# 05 | Supporting work: checked investigation guides

In testing, Llama sounded confident but suggested a PostgreSQL object that wasn't documented. I needed a safer boundary between **possible explanations** and **actions a person might take**.

I kept the model for optional commentary, and added six reviewed investigation guides. Every guide explains when it applies, what to check and what the result might mean.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.scout_playbooks import load_playbooks, select_playbooks

catalogue = load_playbooks()
print("Guides:", len(catalogue))
for guide in catalogue:
    print(guide["id"], "-", guide["title"])

## 1. General versus technical guidance

Some guides cover general methods like building a timeline. Others only apply when a person has confirmed a specific technology. Scout should not assume a company runs PostgreSQL because an unrelated vendor report mentions it.

In [ ]:
initial_issue = "Our checkout API is timing out and database connections are slow."
later_update = initial_issue + " We use PostgreSQL and deployed an update yesterday."

for title, report in [("Initial report", initial_issue), ("After follow-up", later_update)]:
    guides = select_playbooks(report, catalogue)
    print("\n", title)
    for guide in guides:
        print("-", guide["title"])

## 2. Select guides from reported facts

I compare the same incident before and after a follow-up. The selector uses the **user's details**, not Scout's earlier speculation.

In [ ]:
postgre_sql = None
for guide in catalogue:
    if guide["id"] == "PB-POSTGRES-01":
        postgre_sql = guide
        break
assert postgre_sql is not None, "Expected the checked PostgreSQL playbook"
print("Source:", postgre_sql["source_url"])
print("Applicable when:", postgre_sql["applies_when"])
for step in postgre_sql["steps"]:
    if "command" in step:
        print("\nChecked example query:\n", step["command"])
print("\nLimit:", postgre_sql["interpretation"]["unknown"])

## 3. Check the technical instructions

I checked the PostgreSQL example against the official `pg_stat_activity` documentation. The guide uses read-only queries; it does not measure something the source doesn't support.

The user still needs the correct version, permissions and context. **Scout displays commands; it does not run them.**

In [ ]:
from tempfile import TemporaryDirectory
from src.scout_case_store import initialise_database, create_case, add_message, get_messages
from src.scout_conversation import source_snapshot

with TemporaryDirectory() as folder:
    path = Path(folder) / "guides_demo.db"
    initialise_database(path)
    case = create_case("Example checkout problem", path)
    selected = select_playbooks(later_update, catalogue)
    snapshot = source_snapshot([], [], {"model": "demo", "cited_ids": []}, playbooks=selected)
    add_message(case["id"], "user", later_update, db_path=path)
    add_message(case["id"], "assistant", "Let's investigate in order.", evidence=snapshot, db_path=path)
    saved = get_messages(case["id"], path)
    print("Stored steps:", [g["title"] for g in saved[1]["evidence"]["playbooks"]])

## 4. Save the exact plan with each reply

I save the chosen guides inside the case evidence. This keeps a record of **what Scout actually recommended at that time**, even if the guide catalogue changes later.

## 5. What I can and cannot claim

The tests can check guide selection, source links, safe read-only examples and whether the saved plan survives reopening.

They cannot prove that a particular live incident has that root cause, or that every model-generated suggestion is right.

## 6. Do not let Llama change the action order

In a realistic test, Llama wanted to check database capacity first. The reviewed timeline guide was the more appropriate starting step.

I made the checked guide **the visible plan** and saved the original AI draft separately for review. That avoids presenting a model's unverified checklist as the approved procedure.

In [ ]:
from src.scout_playbooks import checked_plan_response
example_text = "Checkout API timeouts and slow PostgreSQL connections after a deployment."
checked_steps = select_playbooks(example_text, catalogue)
model_draft = {"answer": "First check database memory.", "cited_ids": [], "model": "local-demo"}
result = checked_plan_response(model_draft, checked_steps, example_text)
print("Visible priority:\n", result["answer"].split("\n\n")[1])
print("\nOriginal model draft kept separately:\n", result["model_draft"])


## 7. The checked plan should not wait for the model

The original app generated Llama text before showing the checked steps. A direct CPU-only diagnostic measured **7.1 tokens/second**; the older raw-answer test had a **median of 81 seconds**.

I changed the interface to show and save source evidence and checked steps **first**. Optional AI interpretation is a separate request, not a prerequisite.

This does **not** prove that the model itself is faster. Notebook 03 contains the measured timing and caveats.

In [ ]:
# Demonstration of the architectural point — this function is purely local.
# It can prepare the reviewed guidance without generating any AI text.
from time import perf_counter
from src.scout_playbooks import guide_led_answer

local_start = perf_counter()
immediate_plan = guide_led_answer(checked_steps, example_text)
local_plan_seconds = perf_counter() - local_start

print("Reviewed plan generated without an LLM API call.")
print("First recommendation:")
print(immediate_plan.split("\n\n")[1])
print("Local plan construction time: {:.4f}s".format(local_plan_seconds))
print("This is a measured code-cell time when executed, not a complete UI benchmark.")

## 8. What I still need to measure

I have tests for checked-plan ordering, model failure, preserved sources and case memory. I also need a measured **time to first useful plan** on the final Streamlit page and human review of the chosen steps on unseen cases.

The original 15-question Llama scores cannot be transferred to the newer guide-led interface.

## 9. A saved-case regression

An older case failed when I requested optional AI commentary. Its saved technical guide lacked the `applies_to` field that the newer prompt expected.

I made the prompt accept older saved records and added a regression test. **The stored case and checked plan were kept.**

## 10. What I actually saw in Streamlit

In a new 504-timeout case, the app displayed the checked first step without waiting for Llama. I could open its source, add a follow-up and keep the case history.

That was a **manual observation**, not a timed UI benchmark. The first step was still an investigation suggestion, not a confirmed diagnosis.

## Human verification — the extra safeguard

**Q. What if Scout cannot verify an answer?**  
A. It explains **why**, suggests a specific check, and keeps the finding unverified.

**Q. What about Llama's interpretation?**  
A. It gets its **own** notice. Even a source ID doesn't make its claim correct, and the model cannot approve itself.

**Q. What does a human review change?**  
A. A person records **supported**, **not supported** or **uncertain**, with what they checked. Scout appends that decision to the case history without rewriting the reply.

**Q. Does a review certify the whole answer?**  
A. No. It covers the particular finding described and does not establish the live root cause.

## 11. From checked guidance to a completed investigation

I found that adding a new clue wasn't the same as completing a guide. In the final app the investigator records the actual finding against the guide ID. Scout leaves that finding in SQLite and excludes the completed step from the next plan. My demo follows five fictional check results, then records simulated recovery after a rollback performed by the fictional team. Scout does not execute any command or repair. This demonstrates progress and a service recovery conclusion, not proof of a real root cause. Notebook 02 shows the small rule used to filter the next actions.